# Flight Delay Analysis
**Dataset:** 2015 US domestic flights (~5.8M rows)  
**Goal:** Predict arrival/departure delays

## 1. Setup & Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

DATA_DIR = Path('..')

## 2. Load Data Set

Remove all columns not necessary for the cases.
- 'YEAR': 'int16', # alles 2015 -> raus
- 'TAIL_NUMBER' viele missing, irrelevant -> raus
- 'TAXI_OUT': 'float32', # 2% viele missing -> raus
- 'WHEELS_OFF': 'string', # 2% viele missing = TAXI_OFF
- 'AIR_TIME': 'float32', # 2% viele missing = ELAPSED
- 'WHEELS_ON': 'string',  # # 2% viele missing -> raus
- 'TAXI_ON': 'float32', # 2% viele missing = WHEELS_ON
- 'DIVERTED': 'bool', # alle 0 -> raus
- 'CANCELLED': 'bool', # 99.99% sind 0 -> raus
- 'AIRSYSTEM_DELAY'    _> raus weil 80% missing
- 'SECURITY_DELAY'     _> raus weil 80% missing
- 'AIRLINE_DELAY'      _> raus weil 80% missing
- 'LATEAIRCRAFT_DELAY' _> raus weil 80% missing
- 'WEATHER_DELAY'      _> raus weil 80% missing

In [ ]:
# Explicit dtypes cut memory usage roughly in half
FLIGHT_DTYPES = {
    'MONTH': 'int8', 
    'DAY': 'int8',
    'DAY_OF_WEEK': 'int8',
    'AIRLINE': 'string',
    'FLIGHT_NUMBER': 'int32', 
    'ORIGIN_AIRPORT': 'string',
    'DESTINATION_AIRPORT': 'string',
    'SCHEDULED_DEPARTURE': 'string',  
    'DEPARTURE_TIME': 'string',
    'DEPARTURE_DELAY': 'float32',
    'SCHEDULED_TIME': 'float32',
    'ELAPSED_TIME': 'float32',
    'DISTANCE': 'int32',
    'SCHEDULED_ARRIVAL' : 'string', 
    'ARRIVAL_TIME': 'string',
    'ARRIVAL_DELAY': 'float32',
}

print('Loading flights.csv (~592 MB) ...')
flights = pd.read_csv(DATA_DIR / 'flights.csv', dtype=FLIGHT_DTYPES)
airlines = pd.read_csv(DATA_DIR / 'airlines.csv')
airports = pd.read_csv(DATA_DIR / 'airports.csv')

# Keep only columns defined in FLIGHT_DTYPES
flights = flights[list(FLIGHT_DTYPES.keys())]

Loading flights.csv (~592 MB) ...


## 2a. Datenbereinigung

Alle Zeilen löschen, welche einen Null-Wert haben

In [3]:
before = len(flights)

# Drop rows with any missing value
flights = flights.dropna()
after = len(flights)

print(f'dropped {before-after} rows...')

dropped 105071 rows...


Alle Zeilen löschen, wo der Airline IATA Code nicht in der Liste der Airlines ist

In [4]:
valid_airlines = set(airlines['IATA_CODE'])
before = len(flights)
invalid_airlines = flights.loc[~flights['AIRLINE'].isin(valid_airlines), 'AIRLINE'].value_counts()
flights = flights[flights['AIRLINE'].isin(valid_airlines)]
print(f'Airline-Filter: {before - len(flights):,} Zeilen entfernt → {len(flights):,} verbleibend')
print('\nEntfernte Airline IATA-Codes:')
print(invalid_airlines.to_string() if len(invalid_airlines) else '  keine')

Airline-Filter: 0 Zeilen entfernt → 5,714,008 verbleibend

Entfernte Airline IATA-Codes:
  keine


Alle Zeilen löschen, wo der Airport IATA Code nicht in der Liste der Airports ist

In [5]:
valid_airports = set(airports['IATA_CODE'])
before = len(flights)
invalid_origins = flights.loc[~flights['ORIGIN_AIRPORT'].isin(valid_airports), 'ORIGIN_AIRPORT'].value_counts()
invalid_dests = flights.loc[~flights['DESTINATION_AIRPORT'].isin(valid_airports), 'DESTINATION_AIRPORT'].value_counts()
flights = flights[
    flights['ORIGIN_AIRPORT'].isin(valid_airports) &
    flights['DESTINATION_AIRPORT'].isin(valid_airports)
]
print(f'Airport-Filter: {before - len(flights):,} Zeilen entfernt → {len(flights):,} verbleibend')
print('\nEntfernte ORIGIN_AIRPORT-Codes:')
print(invalid_origins.to_string() if len(invalid_origins) else '  keine')
print('\nEntfernte DESTINATION_AIRPORT-Codes:')
print(invalid_dests.to_string() if len(invalid_dests) else '  keine')

Airport-Filter: 482,878 Zeilen entfernt → 5,231,130 verbleibend

Entfernte ORIGIN_AIRPORT-Codes:
ORIGIN_AIRPORT
10397    32509
13930    27566
11298    20586
11292    18077
12892    17628
14771    14094
12266    13091
14107    12884
12889    12649
13487    10552
14747    10361
10721    10091
11433     9882
11057     9537
11618     9508
13204     9006
14869     8738
12953     8447
12478     8255
10821     7991
13232     7598
11278     6759
14679     6184
14100     6065
13303     5929
11259     5873
11697     5753
15304     5125
12191     4681
14057     4539
10693     4450
15016     4295
13796     3994
10423     3871
12173     3761
13495     3646
14893     3598
13198     3536
14908     3524
14831     3457
11042     3202
14492     2984
12264     2979
13342     2631
14683     2594
12339     2361
11066     2178
14122     2122
11193     1903
14843     1858
13830     1814
10140     1805
14635     1756
10800     1753
13891     1734
14027     1734
10529     1705
12451     1701
14524     1628
138

## 2b. Datenerweiterung

Eine neue Spalte hinzufügen 'IS_DELAYED' für alle Arrival-Delays grösser als 15 Minuten (Classifier Label)

In [6]:
flights["IS_DELAYED"] = (flights["ARRIVAL_DELAY"] > 15).astype(int)

### Backup Data Set

In [7]:
out_path = DATA_DIR / 'flights_cut.csv'
flights.to_csv(out_path, index=False)
print(f'Saved {flights.shape[0]:,} rows × {flights.shape[1]} cols → {out_path}')

Saved 5,231,130 rows × 17 cols → ../flights_cut.csv


# 3. Train / Validation Split Data Set

Split original data set into 80% train, 20% validation and then
split train into 75% train and 25% test


In [8]:
# 60% train / 20% test / 20% val
train_test, val = train_test_split(flights, test_size=0.2, random_state=42)
train, test = train_test_split(train_test, test_size=0.25, random_state=42)

train.to_csv(DATA_DIR / 'flights_train.csv', index=False)
test.to_csv(DATA_DIR / 'flights_test.csv', index=False)
val.to_csv(DATA_DIR / 'flights_val.csv', index=False)

total = len(flights)
print(f'Train: {len(train):,} rows ({len(train)/total:.0%})')
print(f'Test:  {len(test):,} rows ({len(test)/total:.0%})')
print(f'Val:   {len(val):,} rows ({len(val)/total:.0%})')

Train: 3,138,678 rows (60%)
Test:  1,046,226 rows (20%)
Val:   1,046,226 rows (20%)


## 3a. Split Validation

Validate the distribution of important features like Airline, Airports and Months over the multiple data sets

In [9]:
DTYPES = {
    "MONTH": "int8",
    "DAY": "int8",
    "DAY_OF_WEEK": "int8",
    "FLIGHT_NUMBER": "int32",
    "DEPARTURE_DELAY": "float32",
    "ARRIVAL_DELAY": "float32",
    "SCHEDULED_DEPARTURE": "string",
    "SCHEDULED_TIME": "float32",
    "SCHEDULED_ARRIVAL": "string",
    "ELAPSED_TIME": "float32",
    "DISTANCE": "int32",
}

DATASETS = {
    "cut": "flights_cut.csv",
    "train": "flights_train.csv",
    "val": "flights_val.csv",
    "test": "flights_test.csv",
}

COMPARE_COLS = ["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "MONTH"]


def load_datasets():
    return {name: pd.read_csv(DATA_DIR / path, dtype=DTYPES) for name, path in DATASETS.items()}


def compare_column(dfs, column, top_n=15):
    dists = {name: df[column].value_counts(normalize=True) for name, df in dfs.items()}
    table = pd.DataFrame(dists).fillna(0.0).sort_values("cut", ascending=False)
    print(f"\n=== {column} (proportion of rows, top {top_n} by 'cut') ===")
    print(table.head(top_n).round(4))
    print(f"\n{column} max abs deviation from 'cut' across datasets:")
    for name in dfs:
        if name == "cut":
            continue
        diff = (table[name] - table["cut"]).abs().max()
        print(f"  cut vs {name}: {diff:.4f}")

dfs = load_datasets()

print("Row counts:")
for name, df in dfs.items():
    print(f"  {name}: {len(df):,}")

for col in COMPARE_COLS:
    compare_column(dfs, col)

Row counts:
  cut: 5,231,130
  train: 3,138,678
  val: 1,046,226
  test: 1,046,226

=== AIRLINE (proportion of rows, top 15 by 'cut') ===
            cut   train     val    test
AIRLINE                                
WN       0.2176  0.2173  0.2188  0.2174
DL       0.1519  0.1521  0.1514  0.1520
AA       0.1217  0.1216  0.1219  0.1217
OO       0.1010  0.1011  0.1006  0.1011
EV       0.0974  0.0973  0.0976  0.0976
UA       0.0883  0.0883  0.0884  0.0884
MQ       0.0492  0.0492  0.0490  0.0491
B6       0.0459  0.0460  0.0458  0.0457
US       0.0371  0.0372  0.0370  0.0370
AS       0.0300  0.0300  0.0299  0.0302
NK       0.0201  0.0201  0.0199  0.0202
F9       0.0157  0.0158  0.0155  0.0156
HA       0.0133  0.0133  0.0134  0.0133
VX       0.0107  0.0107  0.0107  0.0107

AIRLINE max abs deviation from 'cut' across datasets:
  cut vs train: 0.0003
  cut vs val: 0.0011
  cut vs test: 0.0002

=== ORIGIN_AIRPORT (proportion of rows, top 15 by 'cut') ===
                   cut   train     val 

### Show Data Set

In [10]:
print(f'flights: {flights.shape[0]:,} rows × {flights.shape[1]} cols')
flights.head(10)

flights: 5,231,130 rows × 17 cols


,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_TIME,ELAPSED_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,IS_DELAYED
0,1,1,4,AS,98,ANC,SEA,0005,2354,-11.0,205.0,194.0,1448,0430,0408,-22.0,0
1,1,1,4,AA,2336,LAX,PBI,0010,0002,-8.0,280.0,279.0,2330,0750,0741,-9.0,0
2,1,1,4,US,840,SFO,CLT,0020,0018,-2.0,286.0,293.0,2296,0806,0811,5.0,0
3,1,1,4,AA,258,LAX,MIA,0020,0015,-5.0,285.0,281.0,2342,0805,0756,-9.0,0
4,1,1,4,AS,135,SEA,ANC,0025,0024,-1.0,235.0,215.0,1448,0320,0259,-21.0,0
5,1,1,4,DL,806,SFO,MSP,0025,0020,-5.0,217.0,230.0,1589,0602,0610,8.0,0
6,1,1,4,NK,612,LAS,MSP,0025,0019,-6.0,181.0,170.0,1299,0526,0509,-17.0,0
7,1,1,4,US,2013,LAX,CLT,0030,0044,14.0,273.0,249.0,2125,0803,0753,-10.0,0
8,1,1,4,AA,1112,SFO,DFW,0030,0019,-11.0,195.0,193.0,1464,0545,0532,-13.0,0
9,1,1,4,DL,1173,LAS,ATL,0030,0033,3.0,221.0,203.0,1747,0711,0656,-15.0,0


## 4. Classification

Compare classifiers/encodings/scalers for predicting IS_DELAYED, then
retrain the winner on train+val and report a final test-set score.

Note: IS_DELAYED is imbalanced (~81.5% on-time / 18.5% delayed), so the
"always predict on-time" baseline already scores ~0.815 accuracy. Accuracy
alone is reported for comparability with the original script, but
balanced accuracy / F1 / ROC-AUC are tracked too since they're more
informative for this class distribution.

ORIGIN_AIRPORT/DESTINATION_AIRPORT (322 categories each) exceed
HistGradientBoostingClassifier's native categorical cardinality limit
(255), so those two columns get smoothed target-encoding (mean historical
delay rate, fit on train only) for the boosting candidates; AIRLINE (14
categories) stays a native pandas category.

In [ ]:
import time
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MaxAbsScaler,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)

NUM_FEATURES = ["MONTH", "DAY_OF_WEEK", "DISTANCE", "DEPARTURE_HOUR"]
CAT_FEATURES = ["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
HIGH_CARD_FEATURES = ["ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
TARGET_ENCODE_SMOOTHING = 20

DTYPES = {
    "MONTH": "int8",
    "DAY_OF_WEEK": "int8",
    "AIRLINE": "category",
    "ORIGIN_AIRPORT": "category",
    "DESTINATION_AIRPORT": "category",
    "SCHEDULED_DEPARTURE": "int32",
    "DISTANCE": "int32",
    "IS_DELAYED": "int8",
}


def load(filename, frac=None, random_state=42):
    df = pd.read_csv(DATA_DIR / filename, usecols=list(DTYPES), dtype=DTYPES)
    if frac is not None:
        df = df.sample(frac=frac, random_state=random_state)
    df["DEPARTURE_HOUR"] = (df["SCHEDULED_DEPARTURE"] // 100).astype("int8")
    X = df[NUM_FEATURES + CAT_FEATURES]
    y = df["IS_DELAYED"]
    return X, y


def fit_target_encoding(X, y, columns, smoothing=TARGET_ENCODE_SMOOTHING):
    """Smoothed mean-target encoding, fit on training data only."""
    global_mean = y.mean()
    mappings = {}
    for col in columns:
        stats = (
            pd.DataFrame({"cat": X[col].to_numpy(), "y": y.to_numpy()})
            .groupby("cat")["y"]
            .agg(["mean", "count"])
        )
        stats["smoothed"] = (
            stats["mean"] * stats["count"] + global_mean * smoothing
        ) / (stats["count"] + smoothing)
        mappings[col] = stats["smoothed"]
    return mappings, global_mean


def apply_target_encoding(X, mappings, global_mean):
    """Build the boosting feature frame: target-encoded high-cardinality columns
    plus the low-cardinality AIRLINE category and the numeric features."""
    out = X[NUM_FEATURES + ["AIRLINE"]].copy()
    for col, mapping in mappings.items():
        out[f"{col}_DELAY_RATE"] = (
            X[col].map(mapping).fillna(global_mean).astype("float32")
        )
    return out


def evaluate(name, model, X_train, y_train, X_test, y_test):
    t0 = time.time()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = (
        model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    )
    metrics = {
        "name": name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba) if proba is not None else float("nan"),
        "seconds": time.time() - t0,
    }
    return metrics


# Sample for speed while comparing candidates; the winner gets refit on
# the full train+val data at the end.
X_train, y_train = load("flights_train.csv", frac=0.3)
X_test, y_test = load("flights_test.csv")

baseline = max(y_test.mean(), 1 - y_test.mean())
print(f"Majority-class baseline accuracy: {baseline:.4f}\n")

mappings, global_mean = fit_target_encoding(X_train, y_train, HIGH_CARD_FEATURES)
X_train_hgb = apply_target_encoding(X_train, mappings, global_mean)
X_test_hgb = apply_target_encoding(X_test, mappings, global_mean)

onehot = ColumnTransformer(
    [
        ("num", "passthrough", NUM_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
    ]
)
ordinal = ColumnTransformer(
    [
        ("num", "passthrough", NUM_FEATURES),
        (
            "cat",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
            CAT_FEATURES,
        ),
    ]
)

# (name, model, feature_kind) -- feature_kind picks which (X_train, X_test) pair to use.
candidates = []

for scaler_name, scaler in [
    ("StandardScaler", StandardScaler(with_mean=False)),
    ("MaxAbsScaler", MaxAbsScaler()),
    ("NoScaling", None),
]:
    for C in (0.1, 1.0, 10.0):
        steps = [("encode", onehot)]
        if scaler is not None:
            steps.append(("scale", scaler))
        steps.append(("clf", LogisticRegression(max_iter=1000, C=C)))
        name = f"LogisticRegression(C={C}, scaler={scaler_name})"
        candidates.append((name, Pipeline(steps), "raw"))

for n_estimators, max_depth in [(200, 10), (300, 16), (300, None)]:
    name = f"RandomForest(n_estimators={n_estimators}, max_depth={max_depth})"
    model = Pipeline(
        [
            ("encode", ordinal),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    n_jobs=-1,
                    random_state=42,
                ),
            ),
        ]
    )
    candidates.append((name, model, "raw"))

for learning_rate, max_depth, max_iter in [
    (0.1, None, 200),
    (0.05, 8, 400),
    (0.1, 6, 300),
]:
    name = f"HistGradientBoosting(lr={learning_rate}, max_depth={max_depth}, max_iter={max_iter})"
    model = HistGradientBoostingClassifier(
        learning_rate=learning_rate,
        max_depth=max_depth,
        max_iter=max_iter,
        random_state=42,
    )
    candidates.append((name, model, "hgb"))

feature_sets = {"raw": (X_train, X_test), "hgb": (X_train_hgb, X_test_hgb)}

results = []
for name, model, kind in candidates:
    X_tr, X_va = feature_sets[kind]
    metrics = evaluate(name, model, X_tr, y_train, X_va, y_test)
    results.append((metrics, model, kind))

results.sort(key=lambda r: r[0]["accuracy"], reverse=True)

results_df = pd.DataFrame([metrics for metrics, _, _ in results]).set_index("name")
print("\nModel comparison (sorted by accuracy):")
print(results_df.round(4).to_string())

best_metrics, best_model, best_kind = results[0]
print(f"\nBest by accuracy: {best_metrics['name']} (feature kind: {best_kind})")

# Refit the winner on train+val, evaluate once on the held-out test set.
X_full_raw = pd.concat([X_train, X_test], ignore_index=True)
y_full = pd.concat([y_train, y_test], ignore_index=True)

X_val_raw, y_val = load("flights_val.csv")

if best_kind == "hgb":
    full_mappings, full_global_mean = fit_target_encoding(
        X_full_raw, y_full, HIGH_CARD_FEATURES
    )
    X_full = apply_target_encoding(X_full_raw, full_mappings, full_global_mean)
    X_val = apply_target_encoding(X_val_raw, full_mappings, full_global_mean)
else:
    X_full, X_val = X_full_raw, X_val_raw

best_model.fit(X_full, y_full)
y_pred = best_model.predict(X_val)
print(f"\nFinal test accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(classification_report(y_val, y_pred, target_names=["On-time", "Delayed"]))


Majority-class baseline accuracy: 0.8150



/home/ost/work/projects/datahack/data-hack/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/ost/work/projects/datahack/data-hack/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
   


Model comparison (sorted by accuracy):
                                                            accuracy  balanced_accuracy      f1  roc_auc  seconds
name                                                                                                             
HistGradientBoosting(lr=0.1, max_depth=6, max_iter=300)       0.8157             0.5062  0.0289   0.6823  11.6171
HistGradientBoosting(lr=0.05, max_depth=8, max_iter=400)      0.8156             0.5047  0.0220   0.6804  18.5927
HistGradientBoosting(lr=0.1, max_depth=None, max_iter=200)    0.8156             0.5044  0.0204   0.6802  14.1868
RandomForest(n_estimators=300, max_depth=16)                  0.8151             0.5022  0.0111   0.6784  30.2620
RandomForest(n_estimators=200, max_depth=10)                  0.8150             0.5000  0.0000   0.6610  16.9084
LogisticRegression(C=0.1, scaler=NoScaling)                   0.8149             0.5002  0.0011   0.6364  34.3764
LogisticRegression(C=10.0, scaler=NoScaling)    